In [8]:
# Slice Dimension Attributes defined in the plugin. Please check all queries and replace <KEY HERE> with a valid name.
# For example: If slice is defined by Version.[Version Name] and Time.[Month]
# input_df = ibpl Select ([Version].[Version Name].[<KEY HERE>] * [Time].[Month].[<KEY HERE>] * [Item].[Item Number]) on row, ({Measure.[M1], Measure.[M2]}) on column limit 5000;
#                             update <KEY HERE> to valid names
# input_df = ibpl Select ([Version].[Version Name].[CurrentWorkingView] * [Time].[Month].[January] * [Item].[Item Number]) on row, ({Measure.[M1], Measure.[M2]}) on column limit 5000;

_sales_df = "select (Version.[Version Name]*Product.[Product].[196426]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});"
_DPBaseInputs = "select (Version.[Version Name]*Product.[Product].[196426]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});"


# Initialize the O9DataLake with the input parameters and dataframes
# Data can be accessed with O9DataLake.get(<Input Name>)
# Overwritten values will not be reflected in the O9DataLake after initialization

from o9_common_utils.O9DataLake import O9DataLake, ResourceType, DataSource,PluginSetting
sales_df = O9DataLake.register("sales_df",data_source = DataSource.LS, entity_type = ResourceType.IBPL, query = _sales_df,plugin_setting = PluginSetting.Inputs)
DPBaseInputs = O9DataLake.register("DPBaseInputs",data_source = DataSource.LS, entity_type = ResourceType.IBPL, query = _DPBaseInputs,plugin_setting = PluginSetting.Inputs)
O9DataLake.register("Version.[Version Name]", data_source = DataSource.LS, entity_type = ResourceType.IBPL,plugin_setting = PluginSetting.SliceDimension)

O9DataLake.register("out_df1",data_source = DataSource.LS,entity_type = ResourceType.IBPL,plugin_setting = PluginSetting.Outputs)
script_params = O9DataLake.register({}, data_source = DataSource.LS,plugin_setting = PluginSetting.ScriptParam)

In [9]:
O9DataLake.inputs

{'sales_df': {'name': 'sales_df',
  'resource_type': <ResourceType.IBPL: 'ibpl_query'>,
  'data_source': <DataSource.LS: 'liveserver'>,
  'query': 'select (Version.[Version Name]*Product.[Product].[196426]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});',
  'std_count_limit': '200000',
  'df':    Version.[Version Name]  Product.[Product] Time.[FiscalWeek]  \
  0                      S1             196426          W03-2016   
  1      CurrentWorkingView             196426          W03-2016   
  2                      S1             196426          W05-2016   
  3      CurrentWorkingView             196426          W05-2016   
  4                      S1             196426          W08-2016   
  5      CurrentWorkingView             196426          W08-2016   
  6                      S1             196426          W13-2016   
  7    

In [10]:
#getting input from dataLake

sales = O9DataLake.get('sales_df')
dpbaseinputs = O9DataLake.get('DPBaseInputs')
sales.head()

,Version.[Version Name],Product.[Product],Time.[FiscalWeek],SalesAccount.[Account],Location.[Location],DPSellOutUnitsActuals,Mean Pricing Save PCT,Placement Count,Promotion Count,DPSellOutPrice
0,S1,196426,W03-2016,ALL,ALL,1,NaN,NaN,NaN,6.0
1,CurrentWorkingView,196426,W03-2016,ALL,ALL,1,NaN,NaN,NaN,6.0
2,S1,196426,W05-2016,ALL,ALL,2,NaN,NaN,NaN,9.5
3,CurrentWorkingView,196426,W05-2016,ALL,ALL,2,NaN,NaN,NaN,9.5
4,S1,196426,W08-2016,ALL,ALL,1,NaN,NaN,NaN,6.0


In [11]:
import logging
out_df1 = None
logger = logging.getLogger('o9_logger')
logger.info(f'Records count in sales dataframe : {sales.shape}')
logger.info(f'Records count in DPBaseInputs dataframe : {dpbaseinputs.shape}')


2026-02-09 21:23:58,695 - o9_logger - INFO  - Records count in sales dataframe : (14, 10)
2026-02-09 21:23:58,696 - o9_logger - INFO  - Records count in DPBaseInputs dataframe : (14, 10)


In [12]:
#pushing output to the DataLake
O9DataLake.put('out_df1',sales)